# Biomarker Validation — Single Dataset
Classifies using **(A) all genes** and **(B) biomarker-common genes**, with the same classifiers, same CV scheme (5-fold × 10 iter), and same metrics as the original notebook.
Results are stored as **mean ± variance** (not std).

In [1]:
# ============================================================
# CELL 1 — Configuration: choose which GEO dataset to use
# ============================================================
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    accuracy_score, precision_score,
    recall_score, f1_score,
    matthews_corrcoef
)
import warnings
warnings.filterwarnings('ignore')

# ------------------------------------------------------------------
# >>> CHANGE THIS to 'set1' or 'set2' to switch dataset <<<
# ------------------------------------------------------------------
DATASET_CHOICE = 'set1'   # 'set1' = GSE75037  |  'set2' = GSE10072
# ------------------------------------------------------------------

DATASET_INFO = {
    'set1': {
        'label'  : 'GEO Set  — GSE8671 (COAD)',
        # 'expr'   : r"D:\Mcode final__COAD\data\GSE8671_Expression_Matrix_COAD.csv",
        'expr'   : r"GSE117570_expression_matrix.csv",
        'meta'   : r'GSE117570_metadata.csv',
    },
    'set2': {
        'label'  : 'GEO Set  — GSE10072 (LUAD)',
        'expr'   : r'D:\Mcode final__LUAD\data\GSE10072_Expression_Matrix_LUNG.csv',
        'meta'   : r'D:\Mcode final__LUAD\data\GSE10072_Metadata_LUNG.csv',
    },
}

chosen     = DATASET_INFO[DATASET_CHOICE]
LABEL      = chosen['label']

# ------------------------------------------------------------------
# Load biomarkers and dataset
# ------------------------------------------------------------------
biomarkers = pd.read_csv(
    r'D:\Mcode final__COAD\output\extracted_biomarkers.csv'
)['Gene_Symbol'].tolist()

expr_df = pd.read_csv(chosen['expr'], index_col=0)
meta_df = pd.read_csv(chosen['meta'], index_col=0)

# ------------------------------------------------------------------
# Log-transform if raw counts
# ------------------------------------------------------------------
if expr_df.max().max() > 50:
    print(f"{LABEL}: raw counts detected — applying log2(x+1)")
    expr_df = np.log2(expr_df + 1)
else:
    print(f"{LABEL}: already log-transformed")

# ------------------------------------------------------------------
# Build feature matrices
#   X_all    — all genes in the dataset (samples × genes)
#   X_bio    — only genes common with extracted biomarkers
# ------------------------------------------------------------------
common_genes = sorted(set(biomarkers) & set(expr_df.index))
print(f"\nTotal biomarkers             : {len(biomarkers)}")
print(f"Genes in dataset             : {len(expr_df.index)}")
print(f"Biomarker genes in dataset   : {len(common_genes)}")

X_all = expr_df.T                       # all genes
X_bio = expr_df.loc[common_genes].T     # biomarker-common genes

# Labels
y = meta_df.loc[X_all.index, 'Status'].map({'Healthy': 0, 'Tumor': 1})

print(f"\nSamples — Normal: {(y==0).sum()}, Tumor: {(y==1).sum()}, Total: {len(y)}")
print(f"\nFeature set shapes:")
print(f"  All genes       : {X_all.shape}")
print(f"  Biomarker genes : {X_bio.shape}")

GEO Set  — GSE8671 (COAD): raw counts detected — applying log2(x+1)

Total biomarkers             : 172
Genes in dataset             : 6612
Biomarker genes in dataset   : 14

Samples — Normal: 2495, Tumor: 1832, Total: 4327

Feature set shapes:
  All genes       : (4327, 6612)
  Biomarker genes : (4327, 14)


In [ ]:
# ============================================================
# CELL 2 — CV engine (5-fold × 10 iter, same as original)
# Results stored as mean ± VARIANCE (not std)
# ============================================================

METRICS = ['AUC_ROC', 'AUC_PR', 'Accuracy', 'Precision', 'Recall', 'F1', 'MCC']

def make_models(y_arr):
    """Instantiate the same three classifiers used in the original notebook."""
    ratio = (y_arr == 1).sum() / max((y_arr == 0).sum(), 1)
    return {
        'Random Forest': RandomForestClassifier(
            n_estimators=50, class_weight='balanced',
            max_features='sqrt', random_state=42, n_jobs=-1
        ),
        'Linear SVM': SVC(
            kernel='linear', class_weight='balanced',
            probability=True, C=0.1, random_state=42
        ),
        'XGBoost': XGBClassifier(
            n_estimators=50, scale_pos_weight=ratio,
            learning_rate=0.05, max_depth=4,
            eval_metric='logloss', random_state=42,
            use_label_encoder=False,
            n_jobs=-1
        ),
    }


def run_cv(X, y, feature_set_name, dataset_label):
    X_arr = X.values if hasattr(X, 'values') else X
    y_arr = y.values if hasattr(y, 'values') else y

    models = make_models(y_arr)
    iter_scores = {name: {m: [] for m in METRICS} for name in models}

    print(f"\n{'='*62}")
    print(f"CV START — {dataset_label}")
    print(f"Feature set : {feature_set_name}")
    print(f"Models      : {list(models.keys())}")
    print(f"{'='*62}")

    for iteration in range(10):
        print(f"\n🔁 Iteration {iteration+1}/10")

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=iteration)

        for name, model in models.items():
            print(f"   ▶ Model: {name}")

            fold_metrics = {m: [] for m in METRICS}

            for fold, (tr_idx, val_idx) in enumerate(skf.split(X_arr, y_arr), 1):
                print(f"      ⤷ Fold {fold}/5", end='\r')

                X_tr_raw, X_val_raw = X_arr[tr_idx], X_arr[val_idx]
                y_tr, y_val = y_arr[tr_idx], y_arr[val_idx]

                sc = StandardScaler()
                X_tr = sc.fit_transform(X_tr_raw)
                X_val = sc.transform(X_val_raw)

                model.fit(X_tr, y_tr)
                y_prob = model.predict_proba(X_val)[:, 1]
                y_pred = model.predict(X_val)

                fold_metrics['AUC_ROC'].append(roc_auc_score(y_val, y_prob))
                fold_metrics['AUC_PR'].append(
                    average_precision_score(y_val, y_prob))
                fold_metrics['Accuracy'].append(
                    accuracy_score(y_val, y_pred))
                fold_metrics['Precision'].append(
                    precision_score(y_val, y_pred, zero_division=0))
                fold_metrics['Recall'].append(
                    recall_score(y_val, y_pred, zero_division=0))
                fold_metrics['F1'].append(
                    f1_score(y_val, y_pred, zero_division=0))
                fold_metrics['MCC'].append(
                    matthews_corrcoef(y_val, y_pred))

            # Mean over folds
            for m in METRICS:
                iter_scores[name][m].append(np.mean(fold_metrics[m]))

            print(f"      ✔ Completed {name}")

    # Final summary
    summary = {}
    print(f"\n{'='*62}")
    print(f"CV RESULTS — {dataset_label}")
    print(f"Feature set : {feature_set_name}  |  5-fold × 10 iter")
    print(f"{'='*62}")

    for name in models:
        summary[name] = {}
        print(f"\n  {name}")
        for m in METRICS:
            scores = np.array(iter_scores[name][m])
            mean_v = float(np.mean(scores))
            var_v  = float(np.var(scores, ddof=1))

            summary[name][m] = mean_v
            summary[name][m + '_var'] = var_v

            print(f"    {m:12s}: {mean_v:.4f}  ±  {var_v:.6f}  (variance)")

    return summary


# ------------------------------------------------------------------
# Run CV on BOTH feature sets
# ------------------------------------------------------------------
print(f"\n>>> Running CV on ALL GENES  ({X_all.shape[1]} features) <<<")
summary_all = run_cv(X_all, y, 'All Genes', LABEL)

print(f"\n>>> Running CV on BIOMARKER GENES  ({X_bio.shape[1]} features) <<<")
summary_bio = run_cv(X_bio, y, 'Biomarker Genes', LABEL)


>>> Running CV on ALL GENES  (6612 features) <<<

CV START — GEO Set  — GSE8671 (COAD)
Feature set : All Genes
Models      : ['Random Forest', 'XGBoost']

🔁 Iteration 1/10
   ▶ Model: Random Forest
      ✔ Completed Random Forest
   ▶ Model: XGBoost
      ✔ Completed XGBoost

🔁 Iteration 2/10
   ▶ Model: Random Forest
      ✔ Completed Random Forest
   ▶ Model: XGBoost
      ✔ Completed XGBoost

🔁 Iteration 3/10
   ▶ Model: Random Forest
      ✔ Completed Random Forest
   ▶ Model: XGBoost
      ✔ Completed XGBoost

🔁 Iteration 4/10
   ▶ Model: Random Forest
      ✔ Completed Random Forest
   ▶ Model: XGBoost
      ✔ Completed XGBoost

🔁 Iteration 5/10
   ▶ Model: Random Forest
      ✔ Completed Random Forest
   ▶ Model: XGBoost
      ✔ Completed XGBoost

🔁 Iteration 6/10
   ▶ Model: Random Forest
      ✔ Completed Random Forest
   ▶ Model: XGBoost
      ✔ Completed XGBoost

🔁 Iteration 7/10
   ▶ Model: Random Forest
      ✔ Completed Random Forest
   ▶ Model: XGBoost
      ✔ Completed

In [9]:
# ============================================================
# CELL 3 — Save results to CSV (mean ± variance)
# ============================================================

rows = []
for feat_name, summary in [('All_Genes', summary_all), ('Biomarker_Genes', summary_bio)]:
    for model_name, scores in summary.items():
        row = {
            'Dataset'     : LABEL,
            'Feature_Set' : feat_name,
            'Classifier'  : model_name,
        }
        for m in METRICS:
            row[m + '_mean'] = round(scores[m],           6)
            row[m + '_var']  = round(scores[m + '_var'],  8)
        rows.append(row)

results_df = pd.DataFrame(rows)
out_csv    = 'LUAD_scRNA_external_validation.csv'
results_df.to_csv(out_csv, index=False)
print(f"Results saved to: {out_csv}")
results_df

Results saved to: LUAD_scRNA_external_validation.csv


,Dataset,Feature_Set,Classifier,AUC_ROC_mean,AUC_ROC_var,AUC_PR_mean,AUC_PR_var,Accuracy_mean,Accuracy_var,Precision_mean,Precision_var,Recall_mean,Recall_var,F1_mean,F1_var,MCC_mean,MCC_var
0,GEO Set — GSE8671 (COAD),All_Genes,Random Forest,0.961226,2.140000e-06,0.946406,4.250000e-06,0.903028,0.000016,0.902640,0.000024,0.864466,0.000041,0.883001,0.000025,0.800992,0.000070
1,GEO Set — GSE8671 (COAD),All_Genes,XGBoost,0.974222,4.000000e-07,0.968029,8.900000e-07,0.913567,0.000004,0.944644,0.000010,0.845531,0.000010,0.892207,0.000007,0.824112,0.000019
2,GEO Set — GSE8671 (COAD),Biomarker_Genes,Random Forest,0.682541,4.360000e-06,0.573608,7.300000e-06,0.635636,0.000012,0.565795,0.000014,0.602076,0.000098,0.583043,0.000032,0.260772,0.000058
3,GEO Set — GSE8671 (COAD),Biomarker_Genes,XGBoost,0.740680,3.270000e-06,0.628561,1.982000e-05,0.649064,0.000006,0.670903,0.000034,0.338809,0.000031,0.449420,0.000029,0.260340,0.000037


In [1]:
# ============================================================
# External Validation Classification Performance Charts
# Grouped bar chart — Mean values only
# Works for LUAD, HNSC, KIRC (and any additional cohort)
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import os

# ============================================================
# SECTION 1: FILE PATHS — edit cohort name and file path
# ============================================================

cohort_files = {
    "LUAD": r"D:\Mcode final__LUAD\output\LUAD_external_validation.csv",
    "HNSC": r"D:\Mcode final__HNSC\output\HNSC_external_validation.csv",
    "KIRC": r"D:\Mcode final__KIRC\output\KIRC_external_validation.csv",
    # Add more cohorts here if needed:
    # "COAD": "COAD_external_validation.csv",
}

# ============================================================
# SECTION 2: PARAMETERS
# ============================================================

# Metrics to plot (mean columns without '_mean' suffix)
METRICS = ["AUC_ROC", "AUC_PR", "Accuracy", "Precision", "Recall", "F1", "MCC"]

# Metric display labels on x-axis
METRIC_LABELS = ["AUC-ROC", "AUC-PR", "Accuracy", "Precision", "Recall", "F1", "MCC"]

# Colors: 3 classifiers × 2 feature sets = 6 bars per metric
# Order: RF_All, RF_Bio, SVM_All, SVM_Bio, XGB_All, XGB_Bio
COLORS = {
    ("Random Forest", "All_Genes"):      "#1a3a6b",   # dark blue
    ("Random Forest", "Biomarker_Genes"):"#6fa8dc",   # light blue
    ("Linear SVM",    "All_Genes"):      "#b5820a",   # dark gold
    ("Linear SVM",    "Biomarker_Genes"):"#ffe599",   # light gold
    ("XGBoost",       "All_Genes"):      "#7b0000",   # dark red
    ("XGBoost",       "Biomarker_Genes"):"#ea9999",   # light red
}

# Bar display order
BAR_ORDER = [
    ("Random Forest", "All_Genes"),
    ("Random Forest", "Biomarker_Genes"),
    ("Linear SVM",    "All_Genes"),
    ("Linear SVM",    "Biomarker_Genes"),
    ("XGBoost",       "All_Genes"),
    ("XGBoost",       "Biomarker_Genes"),
]

LABEL_MAP = {
    ("Random Forest", "All_Genes"):       "Random Forest (All Genes)",
    ("Random Forest", "Biomarker_Genes"): "Random Forest (Biomarker Genes)",
    ("Linear SVM",    "All_Genes"):       "Linear SVM (All Genes)",
    ("Linear SVM",    "Biomarker_Genes"): "Linear SVM (Biomarker Genes)",
    ("XGBoost",       "All_Genes"):       "XGBoost (All Genes)",
    ("XGBoost",       "Biomarker_Genes"): "XGBoost (Biomarker Genes)",
}

OUTPUT_DIR = "."   # change if you want files saved elsewhere

# ============================================================
# SECTION 3: PLOT FUNCTION
# ============================================================

def plot_cohort(df, cohort_name, geo_id):
    """Generate grouped bar chart for one cohort."""

    n_metrics  = len(METRICS)
    n_bars     = len(BAR_ORDER)
    bar_width  = 0.11
    group_gap  = 0.05                          # extra space between metric groups
    x_centers  = np.arange(n_metrics) * (n_bars * bar_width + group_gap)

    fig, ax = plt.subplots(figsize=(18, 7))

    for i, (clf, fset) in enumerate(BAR_ORDER):
        row = df[(df["Classifier"] == clf) & (df["Feature_Set"] == fset)]
        if row.empty:
            continue

        means  = [row[f"{m}_mean"].values[0] for m in METRICS]
        x_pos  = x_centers + (i - n_bars / 2 + 0.5) * bar_width
        color  = COLORS[(clf, fset)]
        hatch  = "//" if fset == "Biomarker_Genes" else ""

        bars = ax.bar(
            x_pos, means,
            width     = bar_width,
            color     = color,
            hatch     = hatch,
            edgecolor = "white",
            linewidth = 0.6,
            alpha     = 0.92,
            label     = LABEL_MAP[(clf, fset)]
        )

        # Value labels on top of each bar
        for bar, val in zip(bars, means):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.003,
                f"{val:.3f}",
                ha        = "center",
                va        = "bottom",
                fontsize  = 6.2,
                rotation  = 90,
                color     = "#2c2c2c",
                fontweight= "bold"
            )

    # ── Axes formatting ──────────────────────────────────────
    ax.set_xticks(x_centers)
    ax.set_xticklabels(METRIC_LABELS, fontsize=12, fontweight="bold")
    ax.set_ylabel("Score", fontsize=12, fontweight="bold")
    ax.set_ylim(0, 1.18)
    ax.set_yticks(np.arange(0, 1.1, 0.2))
    ax.yaxis.set_tick_params(labelsize=10)

    ax.set_title(
        f"Classification Performance — GEO Set — {geo_id} ({cohort_name})\n"
        f"All Genes vs Biomarker Genes  |  5-fold × 10 iter CV",
        fontsize    = 13,
        fontweight  = "bold",
        pad         = 14
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.yaxis.grid(True, linestyle="--", alpha=0.5, zorder=0)
    ax.set_axisbelow(True)

    # ── Legend ───────────────────────────────────────────────
    handles = [
        mpatches.Patch(
            facecolor = COLORS[key],
            hatch     = "//" if key[1] == "Biomarker_Genes" else "",
            edgecolor = "white",
            label     = LABEL_MAP[key]
        )
        for key in BAR_ORDER
    ]
    ax.legend(
        handles       = handles,
        loc           = "upper right",
        fontsize      = 8.5,
        framealpha    = 0.9,
        ncol          = 2,
        columnspacing = 1.0,
        handlelength  = 1.8
    )

    plt.tight_layout()

    # ── Save ─────────────────────────────────────────────────
    out_png = os.path.join(OUTPUT_DIR, f"{cohort_name}_external_validation.png")
    # out_pdf = os.path.join(OUTPUT_DIR, f"{cohort_name}_external_validation.pdf")

    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    # plt.savefig(out_pdf, bbox_inches="tight")
    plt.close()

    print(f"Saved: {out_png}")
    # print(f"Saved: {out_pdf}")

# ============================================================
# SECTION 4: RUN — loops over all cohorts automatically
# ============================================================

# GEO accession map — add entries if you add more cohorts
GEO_IDS = {
    "LUAD": "GSE10072",
    "HNSC": "GSE75037",
    "KIRC": "GSE40435",
    # "COAD": "GSExxxxx",
}

for cohort, filepath in cohort_files.items():
    df = pd.read_csv(filepath)
    geo_id = GEO_IDS.get(cohort, "")
    plot_cohort(df, cohort, geo_id)

print("\nAll charts generated successfully.")

Saved: .\LUAD_external_validation.png
Saved: .\HNSC_external_validation.png
Saved: .\KIRC_external_validation.png

All charts generated successfully.
